# 5D Cournot analysis through the DTB Ver3 interface

This notebook only selects the game and experiment settings. `experiment.py` performs the complete deterministic DTB run, uses a fixed tangent-coordinate basis, advances the accumulated particles directly, and computes the matched explicit-Euler reference.

[Open in Colab](https://colab.research.google.com/github/sun-mengwei/dtb-colab-experiments/blob/codex%2Fgame-dynamics-dtb/DTB_Ver3/notebooks/cournot_5d_analysis.ipynb)


In [ ]:
from pathlib import Path
import subprocess
import sys

import matplotlib.pyplot as plt

# Find a local checkout. In Colab, obtain the same branch automatically.
candidates = [Path.cwd(), *Path.cwd().parents]
repo_root = next((path for path in candidates if (path / 'DTB_Ver3').is_dir()), None)
if repo_root is None:
    if not Path('/content').is_dir():
        raise FileNotFoundError('Run inside the repository or upload the DTB_Ver3 folder.')
    repo_root = Path('/content/dtb-colab-experiments')
    if not repo_root.exists():
        subprocess.run([
            'git', 'clone', '--depth', '1', '--branch', 'codex/game-dynamics-dtb',
            'https://github.com/sun-mengwei/dtb-colab-experiments.git', str(repo_root),
        ], check=True)
sys.path.insert(0, str(repo_root.resolve()))

from DTB_Ver3 import CournotGame, ExperimentConfig, run_experiment
from DTB_Ver3.utils import plot_cloud_snapshots, plot_diagnostics

package_dir = repo_root / 'DTB_Ver3'


## Game and numerical setup

For $s_i=\sum_{j\ne i}x_j$, the five-player dynamics are

$$
\dot x_i=2b\left(\max\{\mu s_i(1-s_i),0\}-x_i\right),
\qquad b=2,\quad \mu=\frac74.
$$

The settings below reproduce the current 5D deterministic comparison: 3,000 shared initial particles, an ordinary width-16 depth-2 MLP, 64 fixed parameter coordinates, $h=0.005$, and $T=2$.


In [ ]:
game = CournotGame(dim=5, b=2.0, mu=7/4)

config = ExperimentConfig(
    particle_count=3000,
    initial_law='smoothed_uniform',
    smoothing_std=0.02,
    step_size=0.005,
    final_time=2.0,
    snapshot_times=(0.0, 0.5, 1.0, 2.0),
    width=16,
    depth=2,
    activation='tanh',
    basis_size=64,
    svd_rtol=1e-6,
    jacobian_chunk=256,
    seed=0,
    dtype='float64',
    device='auto',
    progress_reports=5,
    output_dir=package_dir / 'results' / 'cournot_5d',
)


## Complete run

This single call initializes the cloud and MLP, fixes the selected tangent coordinates, runs all DTB steps, runs the explicit-Euler reference, reports progress every 20%, and saves the numerical results.


In [ ]:
result = run_experiment(game, config)
result.summary()


## Particle clouds and diagnostics


In [ ]:
pairs = ((1, 2), (2, 3), (3, 4), (4, 5))

plot_cloud_snapshots(
    result.dtb_snapshots,
    coordinate_pairs=pairs,
    title='5D Cournot deterministic DTB',
    color='#176b87',
    output_path=result.output_dir / 'dtb_point_clouds.png',
)
plt.show()

plot_cloud_snapshots(
    result.euler_snapshots,
    coordinate_pairs=pairs,
    title='5D Cournot explicit Euler reference',
    color='#b45309',
    output_path=result.output_dir / 'euler_point_clouds.png',
)
plt.show()

plot_diagnostics(
    result.projection_times,
    result.projection_error,
    result.jacobian_condition,
    output_path=result.output_dir / 'dtb_diagnostics.png',
)
plt.show()


## Package the saved outputs


In [ ]:
import shutil

archive = shutil.make_archive(
    str(result.output_dir),
    'zip',
    root_dir=result.output_dir.parent,
    base_dir=result.output_dir.name,
)
print('Saved result archive:', archive)

try:
    from google.colab import files
except ImportError:
    pass
else:
    files.download(archive)
